# Covering a geometry

`find_extracts_by_geometry` (which `find` and `download` call for a geometry) looks for a set of extracts that covers the geometry without downloading much more than that. This notebook goes through its steps.

In [ ]:
%pip install osmfinder basemaplibre geopandas matplotlib

In [ ]:
import basemaplibre as bml
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from shapely.geometry import box

import osmfinder

In [ ]:
geometry = box(9.4, 47.2, 9.8, 47.6)
result = osmfinder.find_extracts_by_geometry(geometry)

## Steps

In every step, the extract is taken that fits best what is left to cover: the one with the highest intersection over union (IoU). The ones that bring too little are skipped, and so are those that the others turn out to cover as well.

In [ ]:
pd.DataFrame(
    {
        "extract": step.extract.id,
        "iou": step.iou,
        "selected": step.selected,
        "reason": step.reason,
        "coverage": step.cumulative_coverage,
    }
    for step in result.steps
)

Below, the part that each step had left to cover is red, and the extract it looked at is blue, or grey if it was skipped:

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(14, 9), dpi=100, layout="constrained")
for ax in axs.flat:
    ax.set_axis_off()
for ax, step in zip(axs.flat, result.steps, strict=False):
    color = "steelblue" if step.selected else "dimgrey"
    ax.axis((9.0, 10.3, 46.8, 47.9))
    gpd.GeoSeries([step.extract.geometry]).plot(ax=ax, color=color, alpha=0.4, edgecolor=color)
    gpd.GeoSeries([step.geometry_to_cover]).plot(
        ax=ax, color="none", edgecolor="crimson", linewidth=3
    )
    bml.add_basemap(ax, crs=4326, source="OpenFreeMap.Positron", labels="above", attribution_size=6)
    ax.set_title(f"{step.extract.id}\nIoU {step.iou:.3f}, {step.reason}", fontsize=9)
plt.show()

## Threshold

Extracts with the IoU below `geometry_coverage_iou_threshold` (0.01 by default) are left out, even if a part of the geometry stays uncovered: nobody wants a whole country for a few square kilometres. `uncovered_geometry` tells what is missing. Geofabrik has a single extract of Austria, sixty times larger than the box:

In [ ]:
geofabrik = osmfinder.find_extracts_by_geometry(geometry, source="Geofabrik")
print([extract.id for extract in geofabrik.extracts])
print(f"uncovered: {geofabrik.uncovered_geometry.area / geometry.area:.0%}")

With a lower threshold it is taken too:

In [ ]:
complete = osmfinder.find_extracts_by_geometry(
    geometry, source="Geofabrik", geometry_coverage_iou_threshold=0
)
print([extract.id for extract in complete.extracts])
print(f"uncovered: {complete.uncovered_geometry.area / geometry.area:.0%}")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 8), dpi=100, layout="constrained")
for ax, covering in zip(axs, [geofabrik, complete], strict=True):
    ax.axis((9.0, 10.3, 46.8, 47.9))
    covering.plot(ax=ax)
    bml.add_basemap(ax, crs=4326, source="OpenFreeMap.Positron", labels="above", attribution_size=6)
plt.show()

## A single extract

`force_single_result=True` looks for one extract that covers the whole geometry:

In [ ]:
single = osmfinder.find_extracts_by_geometry(geometry, force_single_result=True)
[extract.id for extract in single.extracts]

## How much is there to download

The fewer square kilometres outside of the geometry, the less data to download and to filter out later:

In [ ]:
def total_area(extracts):
    return osmfinder.extracts_to_geodataframe(extracts).to_crs(6933).area.sum() / 1e6


pd.DataFrame(
    {
        name: {
            "extracts": len(covering.extracts),
            "area [km²]": total_area(covering.extracts),
            "uncovered": covering.uncovered_geometry.area / geometry.area,
        }
        for name, covering in {
            "all the sources": result,
            "Geofabrik": geofabrik,
            "Geofabrik, threshold 0": complete,
            "a single extract": single,
        }.items()
    }
).T.round(2)

## Excluding extracts

Extracts that you don't want are passed in `excluded_extracts_ids`, and others take their place:

In [ ]:
other = osmfinder.find_extracts_by_geometry(
    geometry, excluded_extracts_ids={extract.id for extract in result.extracts}
)
[extract.id for extract in other.extracts]